# Train the detector on Kaggle

This notebook is deliberately thin. A run is defined by a config in `configs/` — `train.yaml`
or one rung of `configs/ladder/` — and not by the order these cells happen to be executed in:
the notebook attaches the data, installs the package and calls one command. If you want to
change the schedule, the subset or the anchors, change the config in the repository and commit
it, so that the run stays reproducible from a file rather than from a browser tab.

**Before running:**

1. Add the `petrarodriguez/ls-ssdd-v1-0` dataset as an input. `data.root` in the config
   points at `/kaggle/input/datasets/petrarodriguez/ls-ssdd-v1-0` — Kaggle now mounts a
   dataset input under `/kaggle/input/datasets/<owner>/<slug>`, not the bare
   `/kaggle/input/<slug>` this notebook used to name; if the attachment lands somewhere
   else, the first cell below prints where it actually is and the config is the thing to
   correct.
2. Turn the **Internet** switch on, in the session settings. It is needed once, to fetch the
   COCO backbone weights. Without it, set `model.pretrained: false` and expect much less from
   twelve epochs.
3. Choose the **GPU** accelerator.
4. Set `CONFIG`, in the resume cell below, to the config this session trains — the baseline or
   whichever rung of the ladder is next. It is the one place that names the run: the resume
   cell and the training cell both read it, so there is nothing left to edit twice and nothing
   left to disagree.

The cell after the install puts the clone's `src/` on the path and gives the same path to the training subprocess, so nothing here depends on where `pip install -e` landed — on Kaggle the package has arrived on disk and stayed invisible to the kernel.

The cell after the resume reads the split and refuses an empty held-out half before any GPU
time is spent. It is there because `split_by_scene` does not raise: a `data.images` that no
longer matches the dataset's layout yields nothing to score, over a run that finishes and
reports a number anyway.

**To continue an interrupted run**, attach the previous session's *output* as an input dataset
as well. Kaggle wipes `/kaggle/working` between sessions, so the checkpoint has to come back in
through the door it left by; the third cell copies the last one across before training starts.
Nothing else needs saying — the run reads the directory, sees which epochs are already done and
carries on from the next one.

In [ ]:
!ls /kaggle/input
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# The clone, remade from scratch, and checked before anything below depends on it.
#
# Observed on Kaggle, 2026-08-23, with Persistence left on: /kaggle/working survived the end of
# the session, so `git clone` refused a directory that already existed and printed `fatal:` --
# and a `!` line's failure stops nothing. `pip install` then installed the *previous* session's
# clone, and every cell below would have run against code that was not the code on main. The
# run costs 2h45 and nothing would have said the code was stale.
#
# Removed rather than merely refused: a stale clone is never the thing an operator wanted to
# keep, and telling them the directory is dirty only leaves them to delete it by hand. The
# clone is cheap and `main` is the truth.
import shutil
from pathlib import Path

REPO = Path("/kaggle/working/repo")
if REPO.exists():
    print(f"removing a previous clone at {REPO}")
    shutil.rmtree(REPO)

# Literal paths, no `{...}`: brace substitution in a `!` line belongs to the frontend rather
# than to Python, and this image hands the braces to bash untouched.
!git clone --depth 1 https://github.com/esamoun/dark-vessel-detection.git /kaggle/working/repo

# The clone either produced the package or this cell stops the notebook. A clone that failed --
# Internet left off is the usual reason, and a freshly imported notebook has it off by default
# -- otherwise surfaces as ModuleNotFoundError three cells further down, which reads as a
# packaging problem rather than as the one line that caused it.
assert (REPO / "src" / "darkvessel").is_dir(), (
    f"no src/darkvessel under {REPO}, so the clone did not happen. Internet is off, or the "
    "clone failed for the reason printed just above."
)

# torch and torchvision are already on the image, so the detector extra costs nothing here.
# Not quiet: an install that fails here is the cause of every error in every cell below it, and
# `-q` is what turned that into a hunt through the tracebacks it caused downstream. Left
# non-fatal, unlike the clone above: the package is importable from the clone whatever pip
# does, and this is here only for a dependency the image happens to lack.
!pip install -e "/kaggle/working/repo[detector]"


In [ ]:
# Import from the clone, rather than from wherever the install put the package.
#
# Observed on Kaggle, 2026-08-23: with the repository cloned and `pip install -e` run against
# it, `import darkvessel` still raised ModuleNotFoundError and the `darkvessel` console script
# was not on PATH — the package was on disk and invisible to this kernel. Naming src/ needs no
# install to be right: this is a src layout of pure Python, so the clone is already everything
# the interpreter needs, and the install is left in place only for a dependency the image
# happens to lack.
#
# SRC is named once and read twice — here for this kernel, and by the training cell for the
# child interpreter it starts, which inherits none of this one's state.
import sys

SRC = "/kaggle/working/repo/src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)

In [ ]:
# Bring back the last checkpoint of a previous session, if one is attached as an input.
# Kaggle's working directory does not survive a session; the run's resume does, provided the
# file is put back where the config looks for it.
import re
import shutil
from pathlib import Path

from darkvessel.config import load_config

# The one place a run is named. The training cell below reads it too, so editing this line is
# the whole procedure for switching which rung a session trains or resumes — there is no second
# edit to miss.
CONFIG = "/kaggle/working/repo/configs/train.yaml"

out = load_config(Path(CONFIG))["out"]
working, metrics = Path(out["checkpoints"]), Path(out["metrics"])

# Ordered by epoch, not by path. Sorting paths orders them by the name of the dataset they
# came from, so with two previous outputs attached the older run can win — and it wins quietly.
# Globbing on `working.name` rather than a literal "checkpoints" also narrows this to the rung
# CONFIG names: attaching several previous sessions' outputs at once can no longer surface a
# checkpoint from the wrong rung.
attached = sorted(
    Path("/kaggle/input").glob(f"*/{working.name}/epoch-*.pt"),
    key=lambda path: int(re.findall(r"\d+", path.stem)[-1]),
)

if attached:
    latest = attached[-1]
    working.mkdir(parents=True, exist_ok=True)
    shutil.copy2(latest, working / latest.name)
    # The metrics may legitimately be absent: an epoch's weights land before it is scored, so a
    # session killed in between leaves the checkpoint and no journal. The run scores that epoch
    # from its checkpoint on the way past, which is why this is not an error.
    journal = latest.parent.parent / metrics.name
    if journal.exists():
        metrics.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(journal, metrics)
    print(f"resuming from {latest.name}, metrics {'attached' if journal.exists() else 'missing'}")
else:
    print("no checkpoint attached: this is the first session of the run")

In [ ]:
# The split, read before the GPU hour rather than after it. `split_by_scene` is a pure filter
# and does not raise: point `data.images` at a layout the mirror no longer has — Kaggle's mount
# point and this mirror's directories have both moved once already — and the held-out split
# comes back empty, silently, over a run that trains to completion and reports its one measured
# number over nothing at all. See docs/decisions.md, 2026-08-20.
#
# This cell is what turns that into a stop. It asserts rather than prints, because a printed
# `0` scrolls past a Run All and the training cell starts anyway.
from darkvessel.cli import training_request_from
from darkvessel.detect.dataset import catalogue, split_by_scene

request = training_request_from(load_config(Path(CONFIG)), Path(CONFIG).parent)
training, held_out = split_by_scene(catalogue(request["root"], request["layout"]))
print(f"{len(training)} training tiles, {len(held_out)} held out — expected 6000 and 3000")

assert held_out, (
    "The held-out split is empty, so nothing would be scored. `data.images` in "
    f"{Path(CONFIG).name} names a layout this dataset does not have. Fix the config before "
    "spending the session; see docs/decisions.md, 2026-08-20."
)

In [ ]:
# The run. Plain `subprocess`, and no `!` shell line at all.
#
# Two things a shell line cannot be trusted with here. The command has to name the interpreter
# that holds the package -- the `darkvessel` console script the install declares is not on
# Kaggle's PATH, so a bare `darkvessel` is `command not found`. And `!cmd {expr}` substitution
# is a property of the frontend rather than of Python: this image hands `{sys.executable}` to
# bash untouched, so the run dies on the brace instead of starting. `sys.executable` in an
# argument list is neither of those problems.
#
# PYTHONPATH carries SRC across for the reason the cell above exists at all: the child is a
# fresh interpreter, and this kernel's sys.path does not cross a process boundary.
#
# Popen rather than run(), because a child's stdout in a notebook goes to the kernel's log and
# not to the cell: twelve epochs would print nothing at all until they had finished. This reads
# the pipe and prints each line as it arrives, so an interrupted session still shows how far it
# got -- which is the number the resume depends on.
import os
import subprocess
import sys

with subprocess.Popen(
    [sys.executable, "-m", "darkvessel", "train", "--config", CONFIG],
    env={**os.environ, "PYTHONPATH": SRC},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as run:
    for line in run.stdout:
        print(line, end="")

if run.returncode:
    raise SystemExit(f"training exited with {run.returncode}")

When the session ends, **Save Version** so that `/kaggle/working` becomes an output dataset:
the checkpoints and the metrics file `CONFIG`'s `out.metrics` names are what the next session
resumes from, and that file is what the numbers in the README are copied out of. It is plain
JSON and needs neither torch nor a GPU to read.